# EEG Analysis Example Notebook

This notebook demonstrates interactive EEG analysis using MNE-Python.

In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Configure matplotlib for inline plotting
%matplotlib inline
mne.set_log_level('WARNING')

## 1. Load Sample Data

For demonstration purposes, we'll use MNE's sample dataset.

In [ ]:
# Load sample data (you can replace this with your own data)
sample_data_folder = mne.datasets.sample.data_path()
sample_data_file = sample_data_folder / 'MEG' / 'sample' / 'sample_audvis_raw.fif'

# Load raw data
raw = mne.io.read_raw_fif(sample_data_file, preload=True)
raw.pick_types(meg=False, eeg=True, eog=True, stim=True)
print(raw.info)

## 2. Visualize Raw Data

In [ ]:
# Plot raw data
raw.plot(duration=10, n_channels=30, scalings='auto')

In [ ]:
# Plot power spectral density
raw.compute_psd(fmax=50).plot()

## 3. Preprocess Data

In [ ]:
# Filter data
raw_filtered = raw.copy().filter(l_freq=1.0, h_freq=40.0)
print("Filtering complete")

# Set average reference
raw_filtered.set_eeg_reference('average', projection=False)
print("Re-referencing complete")

## 4. Create Epochs

In [ ]:
# Find events
events = mne.find_events(raw_filtered, stim_channel='STI 014')

# Define event IDs
event_id = {'auditory/left': 1, 'auditory/right': 2, 
            'visual/left': 3, 'visual/right': 4}

# Create epochs
epochs = mne.Epochs(raw_filtered, events, event_id, tmin=-0.2, tmax=0.5,
                   baseline=(None, 0), preload=True)

print(f"Created {len(epochs)} epochs")
print(epochs)

## 5. Visualize Epochs

In [ ]:
# Plot epochs
epochs.plot(n_epochs=10, scalings='auto')

In [ ]:
# Plot epoch image
epochs.plot_image(picks='eeg', combine='mean')

## 6. Compute Evoked Responses

In [ ]:
# Compute evoked responses for each condition
evoked_auditory_left = epochs['auditory/left'].average()
evoked_auditory_right = epochs['auditory/right'].average()
evoked_visual_left = epochs['visual/left'].average()
evoked_visual_right = epochs['visual/right'].average()

print(evoked_auditory_left)

## 7. Visualize Evoked Responses

In [ ]:
# Plot evoked response
evoked_auditory_left.plot(spatial_colors=True, gfp=True)

In [ ]:
# Plot topography
evoked_auditory_left.plot_topomap(times='auto')

In [ ]:
# Joint plot
evoked_auditory_left.plot_joint()

## 8. Compare Conditions

In [ ]:
# Compare all conditions
evokeds = {
    'auditory/left': evoked_auditory_left,
    'auditory/right': evoked_auditory_right,
    'visual/left': evoked_visual_left,
    'visual/right': evoked_visual_right
}

mne.viz.plot_compare_evokeds(evokeds, picks='eeg')

## 9. Time-Frequency Analysis

In [ ]:
# Compute power spectral density over time
freqs = np.arange(6, 40, 2)  # Frequencies of interest
n_cycles = freqs / 2.  # Number of cycles for wavelet

power = mne.time_frequency.tfr_morlet(
    epochs, freqs=freqs, n_cycles=n_cycles, 
    use_fft=True, return_itc=False, average=True
)

# Plot time-frequency representation
power.plot(picks='eeg', baseline=(None, 0), mode='logratio')

## 10. Save Results

Save processed data and figures for later use.

In [ ]:
# Save processed data
# epochs.save('data/processed/my_epochs-epo.fif', overwrite=True)
# evoked_auditory_left.save('data/processed/my_evoked_auditory_left-ave.fif', overwrite=True)

# Save figures
# fig = evoked_auditory_left.plot_joint(show=False)
# fig.savefig('results/evoked_auditory_left.png', dpi=300)

print("Analysis complete!")